# Практическая работа 7. Криптосистемы с открытым ключом
## Криптосистема Эль-Гамаля

In [1]:
'''импорт библиотек'''
import random
import math
from pathlib import Path

### Алгоритмы

In [2]:
def mod(a, k, n):
    b = 1
    a = a % n
    while k > 0:
        if k & 1:
            b = (b * a) % n
        a = (a * a) % n
        k >>= 1
    return b

In [3]:
"""Расширенный алгоритм Евклида.
a*x + b*y = d = НОД(a, b)."""
def ext_gcd(a, b):
    x2, x1 = 1, 0
    y2, y1 = 0, 1
    while b > 0:
        q = a // b
        r = a - q * b
        x = x2 - q * x1
        y = y2 - q * y1
        a, b = b, r
        x2, x1 = x1, x
        y2, y1 = y1, y
    return a, x2, y2

In [4]:
"""Обратный элемент существует, если НОД(a, n) = 1."""
def mod_inverse(a, n):
    d, x, _ = ext_gcd(a % n, n)
    if d != 1:
        raise ValueError(f"Обратного элемента нет: НОД({a}, {n}) = {d}")
    return x % n

In [5]:
"""Тест Миллера-Рабина на простоту"""
def is_probably(n, k=20):
    if n < 4:
        return n in (2, 3)
    if n % 2 == 0:
        return False
    s, t = 0, n - 1
    while t % 2 == 0:
        s += 1
        t //= 2
    for _ in range(k):
        a = random.randint(2, n - 2)
        r = mod(a, t, n)
        if r == 1:
            continue
        found = False
        for _ in range(s):
            if r == n - 1:
                found = True
                break
            r = (r * r) % n
        if not found:
            return False
    return True

In [6]:
"""Генерация случайного простого числа заданной длины."""
def generate(bits):
    while True: # старший бит = 1 (нужная длина), младший = 1 (нечётное)
        n = random.getrandbits(bits) | 1 | (1 << (bits - 1))
        if is_probably(n):
            return n

### Криптосистема

In [7]:
"""Генерация ключевой пары.
Открытый ключ: (p, g, h), закрытый ключ: x."""
def generate_keys(bits=256):
    p = generate(bits)
    g = random.randint(2, p - 2)
    x = random.randint(2, p - 2) # закрытый ключ
    h = mod(g, x, p)
    return (p, g, h), x

In [8]:
"""Зашифровать число m (m < p). Возвращает пару (C1, C2)."""
def encrypt_number(m, pub):
    p, g, h = pub
    k = random.randint(2, p - 2)  # сеансовый ключ
    c1 = mod(g, k, p)
    c2 = (m * mod(h, k, p)) % p
    return c1, c2

In [9]:
"""Расшифровать пару (C1, C2) закрытым ключом x.
m = C2 * (C1^x)^(-1) mod p, т.к. C1^x = g^(kx) = h^k."""
def decrypt_number(c1, c2, p, x):
    s = mod(c1, x, p)
    return (c2 * mod_inverse(s, p)) % p

### блочная схема

In [10]:
"""Разбиваем UTF-8 байты текста на блоки размером (bit_length-1)//8.
Каждый блок интерпретируем как целое число — оно гарантированно < p."""
def text_to_numbers(text, p):
    data = text.encode('utf-8')
    block_size = (p.bit_length() - 1) // 8
    nums = [int.from_bytes(data[i:i + block_size], 'big')
            for i in range(0, len(data), block_size)]
    return nums, len(data)

In [11]:
"""Собираем текст обратно. total_len — исходная длина в байтах
(нужна, потому что последний блок может быть короче)."""
def numbers_to_text(nums, total_len, p):
    block_size = (p.bit_length() - 1) // 8
    data = b''
    for i, n in enumerate(nums):
        size = (total_len - len(data)) if i == len(nums) - 1 else block_size
        data += n.to_bytes(size, 'big')
    return data.decode('utf-8')

### файлы ключа и шифротекст

In [12]:
def save_public(path, pub):
    p, g, h = pub
    Path(path).write_text(f"{p}\n{g}\n{h}\n")

def load_public(path):
    nums = list(map(int, Path(path).read_text().split()))
    return tuple(nums)                      # (p, g, h)

def save_private(path, x):
    Path(path).write_text(f"{x}\n")

def load_private(path):
    return int(Path(path).read_text().strip())

def save_ciphertext(path, length, pairs):
    with open(path, 'w') as f:
        f.write(f"{length}\n")
        for c1, c2 in pairs:
            f.write(f"{c1} {c2}\n")

def load_ciphertext(path):
    lines = Path(path).read_text().strip().split('\n')
    length = int(lines[0])
    pairs = [tuple(map(int, line.split())) for line in lines[1:]]
    return length, pairs

# 5. Атака по выбранному шифртексту
####    Ева перехватывает шифртекст, формирует поддельную пару, подсовывает её Алисе, получает и восстанавливает умножением на t^-1mod(p), закрытый ключ не вычисляется.

In [13]:
def menu_attack():
    pub = load_public("public.key")
    priv = load_private("private.key")
    p, g, h = pub
    src = input("Файл с перехваченным шифртекстом, напр: attack.txt: ") or "attack.txt"
    dst = input("Файл для восстановленного текста, напр: recovered.txt: ") or "recovered.txt"
    t = int(input("Множитель t, напр 2: ") or "2")

    length, pairs = load_ciphertext(src) # Ева формирует поддельный шифртекст
    forged = [(c1, (t * c2) % p) for c1, c2 in pairs] # Алиса расшифровывает
    forged_decrypted = [decrypt_number(c1, c2, p, priv) for c1, c2 in forged] # Ева делит на t по модулю p
    t_inv = mod_inverse(t, p)
    recovered_nums = [(mt * t_inv) % p for mt in forged_decrypted]

    Path(dst).write_text(numbers_to_text(recovered_nums, length, p), encoding='utf-8')
    print(f"Восстановлено {length} байт -> {dst}")

## МЕНЮ

In [14]:
BITS = 256

def menu_generate_keys():
    pub, priv = generate_keys(BITS)
    save_public("public.key", pub)
    save_private("private.key", priv)
    print(f"Ключи сохранены: public.key, private.key (p — {BITS} бит)")

In [15]:
def menu_encrypt():
    pub = load_public("public.key")
    src = input("Файл с открытым текстом: ")
    dst = input("Файл для шифртекста, напр: attack.txt: ") or "attack.txt"
    text = Path(src).read_text(encoding='utf-8')
    nums, length = text_to_numbers(text, pub[0])
    pairs = [encrypt_number(m, pub) for m in nums]
    save_ciphertext(dst, length, pairs)
    print(f"Зашифровано {length} байт в {len(pairs)} блок(ов) -> {dst}")

In [16]:
def menu_decrypt():
    pub = load_public("public.key")
    priv = load_private("private.key")
    src = input("Файл с шифртекстом, напр: attack.txt: ") or "attack.txt"
    dst = input("Файл для открытого текста, напр: plain.txt: ") or "plain.txt"
    length, pairs = load_ciphertext(src)
    nums = [decrypt_number(c1, c2, pub[0], priv) for c1, c2 in pairs]
    Path(dst).write_text(numbers_to_text(nums, length, pub[0]), encoding='utf-8')
    print(f"Расшифровано -> {dst}")

In [17]:
def main():
    actions = {
        '1': menu_generate_keys,
        '2': menu_encrypt,
        '3': menu_decrypt,
        '4': menu_attack,
    }
    while True:
        print("Введите номер пункта меню")
        print("1. Сгенерировать ключи")
        print("2. Зашифровать файл")
        print("3. Расшифровать файл")
        print("4. Атака")
        choice = input("Ввод: ").strip()
        action = actions.get(choice)
        if action:
            try:
                action()
            except Exception as e:
                print(f"Ошибка: {e}")
        else:
            print("Ошибка")

In [ ]:
if __name__ == "__main__":
    main()

Введите номер пункта меню
1. Сгенерировать ключи
2. Зашифровать файл
3. Расшифровать файл
4. Атака


Ввод:  1


Ключи сохранены: public.key, private.key (p — 256 бит)
Введите номер пункта меню
1. Сгенерировать ключи
2. Зашифровать файл
3. Расшифровать файл
4. Атака


Ввод:  2
Файл с открытым текстом:  
Файл для шифртекста, напр: attack.txt:  


Ошибка: [Errno 21] Is a directory: '.'
Введите номер пункта меню
1. Сгенерировать ключи
2. Зашифровать файл
3. Расшифровать файл
4. Атака


...

Зашифровано 79 байт в 3 блок(ов) -> attack.txt
Введите номер пункта меню
1. Сгенерировать ключи
2. Зашифровать файл
3. Расшифровать файл
4. Атака


Ввод:  3
Файл с шифртекстом, напр: attack.txt:  attack.txt
Файл для открытого текста, напр: plain.txt:  plain.txt


Расшифровано -> plain.txt
Введите номер пункта меню
1. Сгенерировать ключи
2. Зашифровать файл
3. Расшифровать файл
4. Атака


Ввод:  4
Файл с перехваченным шифртекстом, напр: attack.txt:  attack.txt
Файл для восстановленного текста, напр: recovered.txt:  recovered.txt
Множитель t, напр 2:  


Восстановлено 79 байт -> recovered.txt
Введите номер пункта меню
1. Сгенерировать ключи
2. Зашифровать файл
3. Расшифровать файл
4. Атака
